Hands-on Task 1: Extend the Basic Pipeline (No LLM)

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 1. Define the extended state structure
class PipelineState(TypedDict):
    text: str
    char_count: int
    word_count: int
    summary: str

# 2. Define the graph nodes
def count_chars(state: PipelineState) -> dict:
    # Counts characters in the input text
    return {"char_count": len(state["text"])}

def count_words(state: PipelineState) -> dict:
    # Counts words in the input text
    words = state["text"].split()
    return {"word_count": len(words)}

def make_summary(state: PipelineState) -> dict:
    # Formats the final output containing metrics
    final_summary = f"Summary: '{state['text']}' — {state['char_count']} characters, {state['word_count']} words."
    return {"summary": final_summary}

# 3. Construct the graph
builder = StateGraph(PipelineState)

# Add nodes
builder.add_node("count", count_chars)
builder.add_node("word_count", count_words)
builder.add_node("summarise", make_summary)

# Define the explicit execution flow
builder.add_edge(START, "count")
builder.add_edge("count", "word_count")
builder.add_edge("word_count", "summarise")
builder.add_edge("summarise", END)

# Compile graph
graph = builder.compile()

# 4. Test the pipeline
initial_state = {"text": "LangGraph makes stateful AI workflows easy!"}
result = graph.invoke(initial_state)

print(result["summary"])

Hands-on Task 2: Build a Review Pipeline with Claude

In [ ]:
from typing import TypedDict
from langchain_anthropic import ChatAnthropic
from langgraph.graph import StateGraph, START, END

# Initialize Claude (ensure ANTHROPIC_API_KEY environment variable is set)
llm = ChatAnthropic(model="claude-3-5-sonnet-20241022", temperature=0.7)

# 1. Define the Review State
class ReviewState(TypedDict):
    product: str
    review: str
    sentiment: str
    reply: str

# 2. Node Functions
def review_node(state: ReviewState) -> dict:
    prompt = f"Write a realistic, brief, one-sentence product review for: {state['product']}. Do not add any filler text."
    response = llm.invoke(prompt)
    return {"review": response.content.strip()}

def sentiment_node(state: ReviewState) -> dict:
    prompt = (
        f"Analyze the sentiment of this product review: '{state['review']}'. "
        f"Respond with exactly one word from these choices: [Positive, Negative, Neutral]."
    )
    response = llm.invoke(prompt)
    # Clean up the output to ensure exact matching
    sentiment = response.content.strip().replace(".", "").capitalize()
    return {"sentiment": sentiment}

def reply_node(state: ReviewState) -> dict:
    prompt = (
        f"You are a professional customer support brand manager. Write a one-line brand response "
        f"to a customer who left a {state['sentiment']} review. "
        f"The user's review was: '{state['review']}'."
    )
    response = llm.invoke(prompt)
    return {"reply": response.content.strip()}

# 3. Workflow Assembly
workflow = StateGraph(ReviewState)

workflow.add_node("review_node", review_node)
workflow.add_node("sentiment_node", sentiment_node)
workflow.add_node("reply_node", reply_node)

workflow.add_edge(START, "review_node")
workflow.add_edge("review_node", "sentiment_node")
workflow.add_edge("sentiment_node", "reply_node")
workflow.add_edge("reply_node", END)

review_app = workflow.compile()

# 4. Execute with Test Input
input_data = {"product": "wireless noise-cancelling headphones"}
final_output = review_app.invoke(input_data)

print(f"Product:   {final_output['product']}")
print(f"Review:    {final_output['review']}")
print(f"Sentiment: {final_output['sentiment']}")
print(f"Response:  {final_output['reply']}")

Hands-on Task 3: Three-Way Conditional Router

In [ ]:
from typing import TypedDict, Literal
from langchain_anthropic import ChatAnthropic
from langgraph.graph import StateGraph, START, END

llm = ChatAnthropic(model="claude-3-5-sonnet-20241022", temperature=0)

# 1. State Definition
class RouterState(TypedDict):
    question: str
    category: str
    response: str

# 2. Persona Node Definitions
def science_node(state: RouterState) -> dict:
    prompt = f"Answer this question as an enthusiastic high school Science Teacher: {state['question']}"
    res = llm.invoke(prompt)
    return {"response": res.content.strip()}

def history_node(state: RouterState) -> dict:
    prompt = f"Answer this question as an authoritative academic Historian: {state['question']}"
    res = llm.invoke(prompt)
    return {"response": res.content.strip()}

def general_node(state: RouterState) -> dict:
    prompt = f"Answer this question as a helpful, balanced general AI assistant: {state['question']}"
    res = llm.invoke(prompt)
    return {"response": res.content.strip()}

# 3. Routing Logic Function
def route_question(state: RouterState) -> Literal["science_node", "history_node", "general_node"]:
    prompt = (
        f"Classify the following user question into one of three categories: 'science', 'history', or 'general'. "
        f"Respond with only the category name.\n\nQuestion: {state['question']}"
    )
    res = llm.invoke(prompt)
    classification = res.content.strip().lower()
    
    if "science" in classification:
        return "science_node"
    elif "history" in classification:
        return "history_node"
    else:
        return "general_node"

# 4. Building the Conditional State Graph
router_builder = StateGraph(RouterState)

# Register nodes
router_builder.add_node("science_node", science_node)
router_builder.add_node("history_node", history_node)
router_builder.add_node("general_node", general_node)

# Apply dynamic routing logic from START node
router_builder.add_conditional_edges(
    START,
    route_question,
    {
        "science_node": "science_node",
        "history_node": "history_node",
        "general_node": "general_node"
    }
)

# Connect worker nodes to structural completion point
router_builder.add_edge("science_node", END)
router_builder.add_edge("history_node", END)
router_builder.add_edge("general_node", END)

router_app = router_builder.compile()

# 5. Testing with Required 4 Questions
test_questions = [
    "Why is the sky blue?",
    "Who was the first emperor of Rome?",
    "What is the best recipe for baking chocolate chip cookies?",
    "How does mRNA work inside a cell vaccine?"
]

print("--- Routing Execution Run ---\n")
for q in test_questions:
    output = router_app.invoke({"question": q})
    print(f"Q: {output['question']}")
    print(f"A:\n{output['response']}")
    print("-" * 40)